## 1. Business Purpose

Weather History Silver cung cấp dữ liệu thời tiết theo giờ tại từng warehouse của FastOrder trong quá khứ.

Dataset này tạo thêm environmental context cho các dữ liệu vận hành như Orders, Delivery và các KPI logistics.

Dữ liệu có thể được sử dụng để:

- Phân tích mối quan hệ giữa điều kiện thời tiết và các vấn đề vận hành như giao hàng chậm, đơn hàng bị hủy hoặc thời gian giao hàng tăng.
- Đánh giá mức độ ảnh hưởng của thời tiết đến các KPI vận hành và logistics.
- Hỗ trợ phân tích nguyên nhân khi hiệu suất giao hàng suy giảm trong những khoảng thời gian có điều kiện thời tiết không thuận lợi.
- Làm cơ sở để xây dựng các phương án vận hành và giao hàng phù hợp hơn trong điều kiện thời tiết xấu.
- Kết hợp với `weather_forecast_hourly` để đánh giá Forecast Accuracy bằng cách so sánh dự báo tại từng snapshot với dữ liệu thời tiết lịch sử tương ứng.

## 2.Overview Data 

In [0]:
import os

account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

endpoint = f"{account_name}.dfs.core.windows.net"

spark.conf.set(
    f"fs.azure.account.auth.type.{endpoint}",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{endpoint}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{endpoint}",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{endpoint}",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{endpoint}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

### Path 

In [0]:
path_response = "weather/open_meteo/historical_forecast/window_start=2026-05-18/window_end=2026-06-16/warehouse_id=WH_HCM/ingestion_id=73310be4-fab6-5318-9efd-dbf07bd21762/response.json"
path_metadata = "weather/open_meteo/historical_forecast/window_start=2026-05-18/window_end=2026-06-16/warehouse_id=WH_HCM/ingestion_id=73310be4-fab6-5318-9efd-dbf07bd21762/metadata.json"

### Metadata

In [0]:
df_metadata = spark.read.format("json").option("multiline", True).load(f"abfss://bronze@fastorderdatalake.dfs.core.windows.net/{path_metadata}")

df_metadata.display()
df_metadata.printSchema()

### response

In [0]:
df_response = spark.read.format("json").option("multiline", True).load(f"abfss://bronze@fastorderdatalake.dfs.core.windows.net/{path_response}")

df_response.display()
df_response.printSchema()

## 3. Thiết kế Silver

### 3.1. Target Dataset

Dataset Silver đích:

`weather_history_hourly`

Dataset này cung cấp dữ liệu thời tiết lịch sử theo giờ tại từng warehouse của FastOrder,
được chuẩn hóa thành dạng bảng để phục vụ phân tích và kết hợp với các dataset downstream
như Orders, Delivery và các KPI vận hành.

---

### 3.2. Grain

Grain của dataset:

**1 dòng = 1 warehouse × 1 giờ thời tiết lịch sử**

Ví dụ:

| warehouse_id | weather_time | temperature_2m |
|---|---|---:|
| WH_HCM | 2026-06-01 07:00 UTC | 30.2 |
| WH_HCM | 2026-06-01 08:00 UTC | 31.0 |
| WH_HN | 2026-06-01 07:00 UTC | 27.4 |

Khác với `weather_forecast_hourly`, Historical không cần lưu nhiều forecast snapshot
cho cùng một thời điểm.

Mục tiêu của `weather_history_hourly` là tạo ra một chuỗi dữ liệu thời tiết lịch sử
duy nhất theo thời gian cho từng warehouse.

---

### 3.3. Candidate Key

Candidate key:

`warehouse_id + weather_time`

Điều này có nghĩa là tại cùng một warehouse và cùng một thời điểm theo giờ,
Silver chỉ nên có một bản ghi thời tiết lịch sử.

`ingestion_id` vẫn được giữ lại để phục vụ lineage và truy vết dữ liệu về Bronze,
nhưng không thuộc business grain của dataset.

---

### 3.4. Các cột trong Silver

#### Thông tin định danh và lineage

| Column | Ý nghĩa |
|---|---|
| `warehouse_id` | Warehouse mà bản ghi thời tiết thuộc về |
| `weather_time` | Thời điểm lịch sử mà dữ liệu thời tiết mô tả |
| `ingestion_id` | Bronze ingestion unit đã tạo ra bản ghi |

#### Thông tin ingestion

| Column | Ý nghĩa |
|---|---|
| `retrieved_at` | Thời điểm FastOrder gọi Historical Forecast API |
| `window_start` | Ngày bắt đầu khoảng thời gian historical được yêu cầu |
| `window_end` | Ngày kết thúc khoảng thời gian historical được yêu cầu |

#### Dữ liệu thời tiết

| Column | Ý nghĩa |
|---|---|
| `temperature_2m` | Nhiệt độ không khí ở độ cao 2 mét |
| `relative_humidity_2m` | Độ ẩm tương đối ở độ cao 2 mét |
| `precipitation` | Lượng mưa |
| `wind_speed_10m` | Tốc độ gió ở độ cao 10 mét |
| `weather_code` | Mã mô tả điều kiện thời tiết |

#### Thông tin vị trí

| Column | Ý nghĩa |
|---|---|
| `requested_latitude` | Latitude FastOrder gửi đến Open-Meteo |
| `requested_longitude` | Longitude FastOrder gửi đến Open-Meteo |
| `response_latitude` | Latitude thực tế mà Open-Meteo sử dụng |
| `response_longitude` | Longitude thực tế mà Open-Meteo sử dụng |

---

### 3.5. Chuẩn hóa thời gian

Toàn bộ timestamp trong Silver được chuẩn hóa về UTC.

`hourly.time` của Open-Meteo được trả về theo timezone:

`Asia/Ho_Chi_Minh`

Quá trình chuẩn hóa:

`hourly.time`
→ parse thành timestamp
→ hiểu theo `Asia/Ho_Chi_Minh`
→ chuyển sang UTC
→ `weather_time`

`retrieved_at` đã biểu diễn thời gian UTC nên chỉ cần parse về Spark `timestamp`.

`window_start` và `window_end` chỉ biểu diễn ngày bắt đầu và ngày kết thúc của
historical ingestion window nên được giữ dưới dạng ngày, không cần chuyển thành timestamp UTC.

---

### 3.6. Quan hệ giữa Bronze và Silver

Mỗi Historical Bronze ingestion hoàn chỉnh gồm:

`response.json`

Chứa các mảng dữ liệu thời tiết theo giờ.

`metadata.json`

Chứa:

- `warehouse_id`
- `ingestion_id`
- thời điểm request
- tọa độ warehouse
- tọa độ response
- `start_date`
- `end_date`
- các thông tin lineage khác

`_SUCCESS`

Xác nhận ingestion unit đã được ghi hoàn chỉnh vào Bronze.

Chỉ những ingestion có `_SUCCESS` mới được phép xử lý sang Silver.

Historical Bronze được ingest theo từng khoảng thời gian.

Ví dụ:

`2026-05-18 → 2026-06-16`

tương ứng khoảng:

`30 ngày × 24 giờ = 720 dòng`

cho một warehouse sau khi flatten dữ liệu hourly.

Do đó:

**1 Historical Bronze ingestion → nhiều dòng Silver hourly**

---

### 3.7. Xử lý trường hợp các historical window bị overlap

Business grain của Silver là:

`warehouse_id + weather_time`

Do đó Silver không được tồn tại nhiều bản ghi cho cùng một warehouse và cùng một
`weather_time`.

Trong lần bootstrap hiện tại, các historical window được chia thành các khoảng liên tiếp
và không overlap.

Tuy nhiên trong tương lai, việc chạy lại backfill hoặc thay đổi khoảng thời gian ingestion
có thể tạo ra các historical window bị chồng lấp.

Ví dụ:

`Window A: 2026-06-01 → 2026-06-30`

`Window B: 2026-06-20 → 2026-07-19`

Khi đó các giờ từ ngày 20/06 đến 30/06 có thể xuất hiện trong cả hai ingestion.

Silver cần phát hiện tình huống này để đảm bảo grain:

`warehouse_id + weather_time`

không bị vi phạm.

Policy xử lý duplicate do overlapping window sẽ được xác định trước khi triển khai
Silver write.

## 4. Transformation

Historical Bronze lưu dữ liệu thời tiết dưới dạng các mảng hourly trong
`response.json` và thông tin ingestion trong `metadata.json`.

Mục tiêu của bước Transformation là chuyển mỗi Historical ingestion thành
các bản ghi thời tiết theo giờ với grain:

**1 dòng = 1 warehouse × 1 historical weather hour**

Luồng transformation:

`Bronze response`
→ `Extract ingestion context`
→ `Flatten hourly arrays`
→ `Attach metadata`
→ `Standardize columns`
→ `Normalize timestamps`
→ `Silver Candidate`

Ở giai đoạn này dữ liệu chưa được ghi xuống Silver.

Silver Candidate sẽ tiếp tục đi qua Data Quality và Validation trước khi được persist.

### 4.1. Extract Ingestion Context

`response.json` không chứa trực tiếp `ingestion_id`.

Do đó, ingestion context được lấy từ metadata của file do Spark cung cấp thông qua:

`_metadata.file_path`

Từ Bronze path:

`.../ingestion_id=<id>/response.json`

pipeline trích xuất `ingestion_id` để có thể liên kết response với `metadata.json`
của cùng ingestion unit.

`_source_file_path` được giữ tạm thời để phục vụ lineage và debugging trong quá trình
Transformation, nhưng sẽ không tồn tại trong Silver output cuối cùng.

In [0]:
BRONZE_HISTORY_ROOT = "weather/open_meteo/historical_forecast"

BRONZE_ABFSS_ROOT = (
    "abfss://bronze@fastorderdatalake.dfs.core.windows.net"
)
SILVER_HISTORY_PATH = (
    "abfss://silver@fastorderdatalake.dfs.core.windows.net/"
    "weather/history_hourly/"
)

In [0]:
from fastorder.storage.adls_client import get_adls_service_client

service_client = get_adls_service_client()

bronze_client = service_client.get_file_system_client(
    "bronze"
)

In [0]:
from fastorder.transformation.silver.weather.forecast_hourly import (
    discover_committed_forecast_ingestions,
    get_processed_forecast_ingestion_ids,
    find_pending_forecast_ingestions,
)
committed_ingestion_paths = (
    discover_committed_forecast_ingestions(
        bronze_client=bronze_client,
        forecast_root=BRONZE_HISTORY_ROOT,
    )
)

print(
    "Committed Bronze ingestions:",
    len(committed_ingestion_paths)
)

In [0]:
processed_ingestion_ids = (
    get_processed_forecast_ingestion_ids(
        spark=spark,
        silver_path=SILVER_HISTORY_PATH,
    )
)

print(
    "Processed Silver ingestions:",
    len(processed_ingestion_ids)
)

In [0]:
pending_ingestion_paths = (
    find_pending_forecast_ingestions(
        committed_ingestion_paths,
        processed_ingestion_ids,
    )
)

print(
    "Pending Forecast ingestions:",
    len(pending_ingestion_paths)
)

In [0]:
import importlib
import fastorder.transformation.silver.weather.history_hourly as history_hourly

importlib.reload(history_hourly)

In [0]:
from fastorder.transformation.silver.weather.history_hourly import (
    extract_history_ingestion_context,
    flatten_history_hourly_arrays,
    attach_history_metadata,
    standardize_history_columns,
    normalize_history_time,
    transform_history_hourly,
    profile_history_data_quality,
    assert_history_data_quality,
    build_history_ingestion_validation_summary,
    profile_history_validation,
    assert_history_validation,
    load_pending_history_bronze
)

In [0]:
if not pending_ingestion_paths:
    print(
        "No pending Forecast ingestions. "
        "Silver is already up to date."
    )

    df_response = None
    df_metadata = None
else:
    df_response, df_metadata = (
        load_pending_history_bronze(
            spark=spark,
            pending_ingestion_paths=pending_ingestion_paths,
            bronze_abfss_root=BRONZE_ABFSS_ROOT,
        )
    )

In [0]:
df_history_context = (
    extract_history_ingestion_context(
        df_response
    )
)

display(
    df_history_context.select(
        "ingestion_id",
        "_source_file_path",
    )
)

### 4.2. Flatten Historical Hourly Arrays

Open-Meteo trả dữ liệu historical hourly dưới dạng nhiều array song song.

Ví dụ:

`hourly.time[i]`

tương ứng với:

- `hourly.temperature_2m[i]`
- `hourly.relative_humidity_2m[i]`
- `hourly.precipitation[i]`
- `hourly.wind_speed_10m[i]`
- `hourly.weather_code[i]`

Do đó các array phải được ghép theo cùng index bằng `arrays_zip()` trước khi sử dụng
`explode()`.

Sau khi flatten:

**1 phần tử hourly → 1 dòng Spark DataFrame**

Đây là bước chuyển dữ liệu từ cấu trúc API-oriented sang cấu trúc tabular phù hợp
với Silver.

In [0]:
df_history_flattened = (
    flatten_history_hourly_arrays(
        df_history_context
    )
)

display(df_history_flattened)

In [0]:
df_history_flattened.groupBy(
    "ingestion_id"
).count().orderBy(
    "ingestion_id"
).show(
    truncate=False
)

### 4.3. Attach Historical Metadata

Sau khi flatten `response.json`, mỗi dòng đã đại diện cho một historical weather hour
nhưng vẫn chưa có đầy đủ business context như:

- `warehouse_id`
- thời điểm API được gọi
- historical window được yêu cầu
- requested coordinates

Các thông tin này nằm trong `metadata.json`.

`response` và `metadata` được liên kết thông qua:

`ingestion_id`

Pipeline sử dụng `LEFT JOIN` từ response sang metadata.

Lý do sử dụng `LEFT JOIN`:

Nếu metadata của một ingestion bị thiếu hoặc không khớp, các hourly record vẫn được giữ
lại trong Silver Candidate để Data Quality có thể phát hiện vấn đề.

Nếu sử dụng `INNER JOIN`, các record có metadata bị thiếu có thể biến mất một cách âm thầm.

In [0]:
df_history_enriched = (
    attach_history_metadata(
        df_response=df_history_flattened,
        df_metadata=df_metadata,
    )
)

display(
    df_history_enriched.limit(20)
)

In [0]:
df_history_enriched.select(
    "warehouse_id",
    "ingestion_id",
    "window_start",
    "window_end",
    "time",
    "temperature_2m",
).show(
    10,
    truncate=False,
)

### 4.4. Standardize Silver Columns

Sau khi kết hợp response và metadata, pipeline chuẩn hóa tên và cấu trúc các column
theo schema của `weather_history_hourly`.

Một số mapping chính:

`requested_at`
→ `retrieved_at`

`time`
→ `weather_time_local`

`request_params.start_date`
→ `window_start`

`request_params.end_date`
→ `window_end`

Ở bước này `weather_time` chưa được chuyển sang UTC.

Column tạm `weather_time_local` được giữ để thể hiện rõ rằng giá trị hiện tại vẫn đang
theo timezone `Asia/Ho_Chi_Minh`.

`_source_file_path` cũng tiếp tục được giữ trong Silver Candidate để hỗ trợ debugging
và lineage trong quá trình validation.

In [0]:
df_history_standardized = (
    standardize_history_columns(
        df_history_enriched
    )
)

df_history_standardized.printSchema()

### 4.5. Normalize Time

Historical weather time từ Open-Meteo được trả về theo timezone:

`Asia/Ho_Chi_Minh`

nhưng raw value không chứa timezone offset.

Ví dụ:

`2026-05-18T08:00`

có nghĩa là:

`2026-05-18 08:00 Asia/Ho_Chi_Minh`

Silver chuẩn hóa timestamp về UTC:

`2026-05-18 08:00 Asia/Ho_Chi_Minh`
→ `2026-05-18 01:00 UTC`

Các field được chuẩn hóa:

- `weather_time_local` → UTC `weather_time`
- `retrieved_at` → Spark timestamp
- `window_start` → Spark date
- `window_end` → Spark date

`window_start` và `window_end` là calendar date mô tả ingestion window nên không cần
timezone conversion.

In [0]:
df_history_normalized = (
    normalize_history_time(
        df_history_standardized
    )
)

display(
    df_history_normalized.select(
        "warehouse_id",
        "window_start",
        "window_end",
        "weather_time",
        "temperature_2m",
    ).limit(20)
)

In [0]:
df_history_normalized.printSchema()

### 4.6. Compose Historical Transformation

Các transformation nhỏ được kết hợp thành một pipeline duy nhất:

`transform_history_hourly()`

Function này nhận:

- Bronze Historical `response.json`
- Bronze Historical `metadata.json`

và trả về:

**Historical Silver Candidate**

Silver Candidate đã có schema và grain gần với Silver đích nhưng chưa được persist.

Dữ liệu vẫn phải đi qua:

`Data Quality`
→ `Validation`
→ `Silver Write`

trước khi trở thành dữ liệu Silver chính thức.

In [0]:
df_history_candidate = (
    history_hourly.transform_history_hourly(
        df_response=df_response,
        df_metadata=df_metadata,
    )
)

### 4.7. Kiểm tra sơ bộ Silver Candidate

In [0]:
display(
    df_history_candidate
    .groupBy(
        "ingestion_id",
        "warehouse_id",
        "window_start",
        "window_end",
    )
    .count()
    .orderBy(
        "warehouse_id",
        "window_start",
    )
)

In [0]:
actual_ingestion_count = (
    df_history_candidate
    .select("ingestion_id")
    .distinct()
    .count()
)

print(
    "Expected ingestions:",
    len(pending_ingestion_paths)
)

print(
    "Actual ingestions:",
    actual_ingestion_count
)

## 5. Data Quality

Silver Candidate đã có cấu trúc gần với dataset đích nhưng chưa được phép ghi xuống Silver.

Bước Data Quality kiểm tra xem các giá trị trong dữ liệu có hợp lệ hay không.

Các nhóm kiểm tra chính:

### Identity và lineage

- `warehouse_id` không được NULL
- `ingestion_id` không được NULL
- `weather_time` không được NULL

### Ingestion context

- `retrieved_at` không được NULL
- `window_start` không được NULL
- `window_end` không được NULL
- `window_start` không được lớn hơn `window_end`

### Weather measurements

- `temperature_2m` không được NULL
- `relative_humidity_2m` không được NULL
- `precipitation` không được NULL
- `wind_speed_10m` không được NULL
- `weather_code` không được NULL

Các range rule:

- `0 <= relative_humidity_2m <= 100`
- `precipitation >= 0`
- `wind_speed_10m >= 0`

### Business grain

Grain của `weather_history_hourly` là:

`warehouse_id + weather_time`

Do đó không được tồn tại nhiều hơn một bản ghi cho cùng warehouse và cùng historical hour.

Nếu duplicate xuất hiện, pipeline sẽ fail thay vì tự động deduplicate.

Điều này giúp tránh việc âm thầm lựa chọn một record khi chưa có policy rõ ràng cho
trường hợp historical ingestion window bị overlap.

### 5.1. Profile Data Quality

Data Quality profiling tính số lượng record vi phạm từng rule nhưng chưa dừng pipeline.

Mục đích của bước profiling là cho phép quan sát toàn bộ vấn đề dữ liệu trước khi
thực hiện fail-fast assertion.

Một Data Quality result hoàn toàn hợp lệ sẽ có tất cả metric bằng `0`.

In [0]:
dq_result = (
    history_hourly.profile_history_data_quality(
        df_history_candidate
    )
)

dq_result

### 5.2. Data Quality Assertion

Sau khi profiling, pipeline kiểm tra toàn bộ Data Quality metrics.

Nếu bất kỳ metric nào khác `0`, Silver Candidate không được phép tiếp tục đến bước
Silver Write.

Pipeline sử dụng fail-fast thay vì tự động:

- fill NULL
- clamp giá trị
- drop record
- drop duplicate

Việc sửa hoặc loại bỏ dữ liệu chỉ được thực hiện khi có policy rõ ràng và có thể giải thích
được về mặt dữ liệu hoặc business.

In [0]:
history_hourly.assert_history_data_quality(
    dq_result
)

## 6. Validation

Data Quality kiểm tra dữ liệu có hợp lệ về mặt giá trị và business grain hay không.

Validation kiểm tra pipeline có xử lý đầy đủ dữ liệu được kỳ vọng hay không.

Đối với Historical Forecast, số lượng row của mỗi ingestion phụ thuộc vào historical window:

`expected_rows = (window_end - window_start + 1) × 24`

Validation kiểm tra:

- Tất cả ingestion được yêu cầu xử lý đều xuất hiện trong Silver Candidate.
- Không xuất hiện ingestion ngoài tập dữ liệu được yêu cầu.
- Mỗi ingestion tạo ra đúng số hourly row theo historical window.
- Mỗi ingestion có đủ số `weather_time` khác nhau được kỳ vọng.
- Khoảng thời gian thực tế bắt đầu và kết thúc đúng với historical window.
- Tổng số row thực tế bằng tổng số row kỳ vọng.

Validation chỉ được thực hiện sau khi Data Quality đã PASS.

### 6.1. Per-Ingestion Validation Summary

Trước khi tạo kết quả Validation tổng hợp, pipeline xây dựng summary cho từng
Historical ingestion.

Summary so sánh:

- số row thực tế
- số historical hour khác nhau
- thời điểm bắt đầu thực tế
- thời điểm kết thúc thực tế

với giá trị kỳ vọng được tính từ `window_start` và `window_end`.

Khác với Forecast, Historical không sử dụng một số row cố định cho mọi ingestion.

In [0]:
df_validation_summary = (
    history_hourly
    .build_history_ingestion_validation_summary(
        df_history_candidate
    )
)

display(
    df_validation_summary
    .orderBy(
        "warehouse_id",
        "window_start",
    )
)

### 6.2. Profile Validation

Validation profiling tổng hợp mức độ đầy đủ của Historical Silver Candidate.

Một kết quả hợp lệ cần đảm bảo:

- `pending_ingestion_count = actual_ingestion_count`
- `expected_total_rows = actual_total_rows`
- không có missing ingestion
- không có unexpected ingestion
- không có ingestion sai row count
- không có ingestion thiếu historical hour
- không có ingestion sai time range

In [0]:

validation_result = (
    history_hourly.profile_history_validation(
        df=df_history_candidate,
        pending_ingestion_paths=
            pending_ingestion_paths,
    )
)

validation_result

### 6.3. Validation Assertion

Sau khi profiling, pipeline kiểm tra toàn bộ Validation metrics.

Validation FAIL nếu:

- tổng số row không khớp kỳ vọng
- có ingestion bị thiếu
- xuất hiện ingestion ngoài tập được yêu cầu
- một ingestion có số row không đúng với historical window
- một ingestion không có đủ số historical hour khác nhau
- actual time range không khớp với historical window

Nếu Validation FAIL, dữ liệu không được ghi xuống Silver.

In [0]:
history_hourly.assert_history_validation(
    validation_result
)

print(
    "Historical Forecast Validation: PASS"
)